In [5]:
import requests 

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []
c=0
d=0
for course in documents_raw:
    c = c+1
    course_name = course['course']
    print(course_name)
    for doc in course['documents']:
        d = d + 1
        doc['course'] = course_name
        documents.append(doc)

data-engineering-zoomcamp
machine-learning-zoomcamp
mlops-zoomcamp


In [30]:
print(c,d)
documents[900]

3 948


{'text': "Problem: I tried to run starter notebook on pipenv environment but had issues with no output on prints. \nI used scikit-learn==1.2.2 and python==3.10\nTornado version was 6.3.2\n\nSolution: The error you're encountering seems to be a bug related to Tornado, which is a Python web server and networking library. It's used by Jupyter under the hood to handle networking tasks.\nDowngrading to tornado==6.1 fixed the issue\nhttps://stackoverflow.com/questions/54971836/no-output-jupyter-notebook",
 'section': 'Module 4: Deployment',
 'question': 'Pipenv with Jupyter no output',
 'course': 'mlops-zoomcamp'}

In [9]:
import minsearch

index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

In [10]:
from openai import OpenAI
import os
api_key = os.getenv("OPENROUTER_API_KEY")
api_key

'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'

In [17]:
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=api_key,
)

In [11]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5
    )

    return results

In [12]:
def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT: 
{context}
""".strip()

    context = ""
    
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

In [14]:
def llm(prompt):
    response = client.chat.completions.create(
    model="z-ai/glm-4.5-air:free", 
    messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [15]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [18]:
rag('how do I run kafka?')

'Based on the context provided, here\'s how to run Kafka:\n\n1. For Java Kafka applications:\n   - In the project directory, run:\n   ```\n   java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n   ```\n\n2. For Python Kafka applications:\n   - First create a virtual environment:\n   ```\n   python -m venv env\n   ```\n   - Activate it (on macOS/Linux):\n   ```\n   source env/bin/activate\n   ```\n   - On Windows, use:\n   ```\n   env/Scripts/activate\n   ```\n   - Install requirements:\n   ```\n   pip install -r ../requirements.txt\n   ```\n   - Remember to activate the virtual environment each time you need to run the code\n   - Deactivate when done:\n   ```\n   deactivate\n   ```\n\n3. If you encounter a permission error with build.sh:\n   - In the /docker/spark directory, run:\n   ```\n   chmod +x build.sh\n   ```\n\n4. If you get a "ModuleNotFoundError: No module named \'kafka.vendor.six.moves\'" error:\n   - Install kafka-python-ng inst

In [19]:
from qdrant_client import QdrantClient, models

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-29 03:10:27.440331321 [W:onnxruntime:Default, device_discovery.cc:132 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename ""5620e0c7-8062-4dce-aeb7-520c7ef76171"" dit not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [20]:
qd_client = QdrantClient("http://localhost:6333")

In [22]:
EMBEDDING_DIMENSIONALITY = 512
model_handle = "jinaai/jina-embeddings-v2-small-en"

In [23]:
collection_name = "zoomcamp-faq"

In [24]:
qd_client.delete_collection(collection_name=collection_name)

False

In [26]:
qd_client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=EMBEDDING_DIMENSIONALITY,
        distance=models.Distance.COSINE
    )
)

True

In [27]:
qd_client.create_payload_index(
    collection_name=collection_name,
    field_name="course",
    field_schema="keyword"
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [31]:
documents[0]

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [32]:
points = []

for i,doc in enumerate(documents):
    text = doc['question'] + '::' + doc['text']
    vector = models.Document(text=text, model=model_handle)
    point = models.PointStruct(
        id=i,
        vector=vector,
        payload=doc
    )
    points.append(point)

In [33]:
qd_client.upsert(
    collection_name=collection_name,
    points=points
)

Fetching 5 files: 100%|███████████████████████████████████████████████| 5/5 [00:03<00:00,  1.51it/s]


UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [34]:
question = 'I just discovered the course. Can I still join it?'

In [35]:
def vector_search(question):
    print('vector_search is used')

    course = 'data-engineering-zoomcamp'
    query_points = qd_client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=question,
            model=model_handle
        ),
        query_filter = models.Filter(
            must=[
                models.FieldCondition(
                    key="course",
                    match=models.MatchValue(value=course)
                )
            ]
        ),
        limit=5,
        with_payload=True
    )

    results = []
    print(query_points)
    for point in query_points.points:
        result.append(point.payload)

    return results

In [36]:
def rag(query):
    search_results = vector_search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [ ]:
rag('how do I run kafka?')